# الدرس 10 - وكلاء الذكاء الاصطناعي في الإنتاج

في هذا الدرس ستتعلم **أنماط الإنتاج** لوكلاء الذكاء الاصطناعي باستخدام إطار عمل وكيل مايكروسوفت (بايثون). نغطي:

- **المراقبة** — إضافة التوقيت وتسجيل التفاعلات مع الوكيل
- **التقييم** — استخدام وكيل التقييم لتسجيل جودة الاستجابة
- **إدارة التكاليف** — استراتيجيات تحسين الرموز واختيار النموذج

السيناريو هو **وكيل سفر** يساعد المستخدمين في تخطيط الرحلات، مع وجود طبقات مراقبة وتقييم فوقه.


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## اعتبارات الإنتاج

يتطلب نقل وكلاء الذكاء الاصطناعي من النماذج الأولية إلى الإنتاج اهتمامًا دقيقًا بثلاثة أعمدة:

1. **الرصد** — تحتاج إلى رؤية لما يفعله الوكيل، ومدة كل عملية، والأدوات التي يستخدمها. بدون التتبع والتسجيل، يصبح تصحيح مشكلات الإنتاج شبه مستحيل.

2. **التقييم** — تضمن الفحوصات الآلية للجودة بقاء استجابات الوكيل دقيقة وكاملة ومفيدة مع مرور الوقت. يمكن لوكيل التقييم أن يقيم الاستجابات وفقًا لمعايير محددة.

3. **إدارة التكلفة** — يؤثر استخدام الرموز بشكل مباشر على التكلفة. تساعد استراتيجيات مثل تحسين الأوامر، اختيار النموذج، والتخزين المؤقت في الحفاظ على النفقات تحت السيطرة دون التضحية بالجودة.


## بناء وكيل قابل للمراقبة

نحن نعرّف أدوات السفر ونغلف استدعاء الوكيل بالتوقيت حتى نتمكن من مراقبة الكمون. في بيئة الإنتاج، ستدمج مع OpenTelemetry أو نظام تتبع مماثل.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## أنماط التقييم

نمط شائع في الإنتاج هو استخدام وكيل ثانٍ كـ **المُقيّم**. يقوم المُقيّم بتقييم استجابة الوكيل الأساسي وفقًا لمعايير محددة مسبقًا مثل الشمولية والدقة والفائدة.

هذا يمكّن من:
- بوابات جودة آلية قبل وصول الردود إلى المستخدمين
- اكتشاف التراجع عندما تتغير المطالبات أو النماذج
- مراقبة مستمرة لأداء الوكيل مع مرور الوقت


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## استراتيجيات إدارة التكاليف

التحكم في التكاليف أمر حاسم لوكلاء الذكاء الاصطناعي للإنتاج. فيما يلي الاستراتيجيات الرئيسية:

| الاستراتيجية | الوصف |
|---|---|
| **تحسين التعليمات البرمجية** | اجعل تعليمات النظام موجزة. قم بإزالة السياق المكرر لتقليل رموز الإدخال. |
    "| **اختيار النموذج** | استخدم نماذج أصغر وأرخص (مثل GPT-5-mini) للمهام البسيطة مثل التصنيف أو الاستخراج، واحتفظ بالنماذج الأكبر للمنطق المعقد. |\n",
| **التخزين المؤقت** | خزّن نتائج الأدوات والاستعلامات المتكررة لتجنب مكالمات API المكررة. |
| **ميزانيات الرموز** | حدد حدود `max_tokens` لمنع الردود الطويلة غير المتوقعة. |
| **التجميع** | جمع عدة استعلامات مستخدمين في مكالمة API واحدة حيثما أمكن. |

في الممارسة، تعمل الطريقة متعددة المستويات بشكل جيد: وجه الطلبات البسيطة إلى نموذج سريع ورخيص واحتفظ بالاستعلامات المعقدة للنموذج الأكثر قدرة.


## ملخص

في هذا الدرس تعلمت كيفية:

1. **إضافة الرصد** لتفاعلات الوكيل مع التوقيت والتسجيل، مما يؤسس لقاعدة تتبع ومراقبة.
2. **تقييم ردود الوكيل** تلقائيًا باستخدام وكيل مقيم يقوم بتسجيل الاكتمال والدقة والفائدة.
3. **إدارة التكاليف** من خلال تحسين الطلبات، اختيار النموذج، التخزين المؤقت، وميزانيات الرموز.

تساعد هذه الأنماط الإنتاجية على ضمان أن وكلاء الذكاء الاصطناعي الخاصين بك موثوقون وقابلون للقياس وفعالون من حيث التكلفة على نطاق واسع.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
